# 03 — Modelling

**Input:** processed parquet files from `data/processed/`
**Output:** out-of-fold predictions for both models (saved to `data/processed/`), final trained LightGBM model (saved to `models/`), and a CV summary table.

## Models

Two models, deliberately chosen to span the interpretability-performance trade-off that credit teams actually navigate:

| Model | Role | Notes |
|---|---|---|
| **Logistic regression** | Baseline | Regulated-lending standard. Interpretable, auditable, slow to drift. We expect LightGBM to beat it on ranking metrics — that gap quantifies the lift we'd be trading interpretability for. |
| **LightGBM** | Production candidate | Industry-standard gradient boosting for credit scoring. Handles NaN natively. Tuned via Optuna (40 trials, 30 min cap). |

## CV protocol

- **Stratified 5-fold**, fixed random seed (42). Stratified because of 8.07% class imbalance — random splits could produce folds with materially different positive rates.
- **All preprocessing inside the fold loop.** Scaling for LR is fitted on each train fold, not globally. This prevents the validation fold's variance from leaking into the train fold's scaling.
- **Out-of-fold predictions saved** — these are the predictions we evaluate in `04_evaluation.ipynb`. Every applicant in the train set gets a prediction made by a model that didn't see them during training.

## Class imbalance handling

- **LR:** `class_weight="balanced"` (sklearn re-weights the loss by inverse class frequency)
- **LightGBM:** `scale_pos_weight=11.4` (the inverse of the positive class rate, ~1/0.0807)


## 1. Setup and load processed data

In [1]:
import sys
from pathlib import Path
import time
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score

import lightgbm as lgb
import optuna

PROCESSED_DIR = REPO_ROOT / "data" / "processed"
MODELS_DIR = REPO_ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
N_FOLDS = 5

print(f"sklearn ready, lightgbm {lgb.__version__}, optuna {optuna.__version__}")

sklearn ready, lightgbm 4.5.0, optuna 4.0.0


In [2]:
# Load the two parquet variants
train_lgb = pd.read_parquet(PROCESSED_DIR / "application_train_processed.parquet")
train_lr = pd.read_parquet(PROCESSED_DIR / "application_train_imputed.parquet")

# Sanity check: same row counts, same TARGET values
assert len(train_lgb) == len(train_lr)
assert (train_lgb["TARGET"].values == train_lr["TARGET"].values).all()

# Feature columns: same in both, just NaN-handling differs
FEATURE_COLS = [c for c in train_lgb.columns if c not in ("TARGET", "SK_ID_CURR")]
y = train_lgb["TARGET"].values
ids = train_lgb["SK_ID_CURR"].values

X_lgb = train_lgb[FEATURE_COLS].values  # has NaN
X_lr = train_lr[FEATURE_COLS].values    # imputed

print(f"X shape: {X_lgb.shape}")
print(f"Feature count: {len(FEATURE_COLS)}")
print(f"Positive rate: {y.mean():.4f} ({y.sum():,} of {len(y):,})")

X shape: (307507, 264)
Feature count: 264
Positive rate: 0.0807 (24,825 of 307,507)


## 2. Cross-validation splitter

One splitter object used by both models so the folds are identical. This lets us compare per-fold OOF scores apples-to-apples.

In [3]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

# Materialise the splits so we can reuse them for both models
splits = list(skf.split(X_lgb, y))

# Quick check: positive rate per fold (should be near 0.0807 in all folds)
for i, (tr, va) in enumerate(splits):
    print(f"Fold {i+1}: train n={len(tr):,} pos_rate={y[tr].mean():.4f} | "
          f"val n={len(va):,} pos_rate={y[va].mean():.4f}")

Fold 1: train n=246,005 pos_rate=0.0807 | val n=61,502 pos_rate=0.0807
Fold 2: train n=246,005 pos_rate=0.0807 | val n=61,502 pos_rate=0.0807
Fold 3: train n=246,006 pos_rate=0.0807 | val n=61,501 pos_rate=0.0807
Fold 4: train n=246,006 pos_rate=0.0807 | val n=61,501 pos_rate=0.0807
Fold 5: train n=246,006 pos_rate=0.0807 | val n=61,501 pos_rate=0.0807


**Sanity check:** validation positive rate should be ~0.0807 (the overall rate) in every fold, give or take 0.0001. Stratification is working if these are all close.

## 3. Logistic regression baseline

For LR we use the median-imputed matrix and fit a `StandardScaler` inside each fold. The scaler is fitted on train data only — never on validation data.

`solver="saga"` is the right choice for high-dimensional sparse-after-encoding data like ours. `max_iter=1000` because SAGA can be slow to converge on this kind of matrix.

In [4]:
def fit_lr_cv(X, y, splits):
    """Fit LR per fold, return OOF predictions and per-fold AUCs."""
    oof = np.zeros(len(y))
    fold_aucs = []
    fold_aps = []
    t0 = time.time()
    
    for i, (tr_idx, va_idx) in enumerate(splits):
        X_tr, X_va = X[tr_idx], X[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]
        
        # Scale inside the fold to avoid leakage
        scaler = StandardScaler()
        X_tr_s = scaler.fit_transform(X_tr)
        X_va_s = scaler.transform(X_va)
        
        model = LogisticRegression(
            class_weight="balanced",
            solver="saga",
            max_iter=1000,
            C=1.0,
            random_state=SEED,
            n_jobs=-1,
        )
        model.fit(X_tr_s, y_tr)
        proba = model.predict_proba(X_va_s)[:, 1]
        
        oof[va_idx] = proba
        auc = roc_auc_score(y_va, proba)
        ap = average_precision_score(y_va, proba)
        fold_aucs.append(auc)
        fold_aps.append(ap)
        print(f"  Fold {i+1}: AUC={auc:.4f}  AP={ap:.4f}  "
              f"elapsed={time.time()-t0:.0f}s")
    
    return oof, fold_aucs, fold_aps

print("Fitting LR baseline...")
lr_oof, lr_aucs, lr_aps = fit_lr_cv(X_lr, y, splits)

print(f"\nLR CV ROC-AUC: {np.mean(lr_aucs):.4f} (+/- {np.std(lr_aucs):.4f})")
print(f"LR CV PR-AUC:  {np.mean(lr_aps):.4f} (+/- {np.std(lr_aps):.4f})")
print(f"LR overall OOF ROC-AUC: {roc_auc_score(y, lr_oof):.4f}")
print(f"LR overall OOF PR-AUC:  {average_precision_score(y, lr_oof):.4f}")

Fitting LR baseline...
  Fold 1: AUC=0.7431  AP=0.2270  elapsed=273s
  Fold 2: AUC=0.7564  AP=0.2318  elapsed=537s
  Fold 3: AUC=0.7521  AP=0.2296  elapsed=804s
  Fold 4: AUC=0.7533  AP=0.2322  elapsed=1070s


KeyboardInterrupt: 

**Expected:** LR CV ROC-AUC in the 0.72-0.76 range. That's the baseline number — LightGBM should beat it materially. If LR scores below 0.70, something's wrong (likely a scaling or convergence issue).

**Actual:**
  Fold 1: AUC=0.7431  AP=0.2270  elapsed=273s
  Fold 2: AUC=0.7564  AP=0.2318  elapsed=537s
  Fold 3: AUC=0.7521  AP=0.2296  elapsed=804s
  Fold 4: AUC=0.7533  AP=0.2322  elapsed=1070s

## 4. LightGBM baseline (untuned)

Before tuning, we fit LightGBM with sensible defaults to establish the untuned reference. This is the number we compare tuning against — tuning that doesn't beat the default isn't worth the time.

LightGBM uses the NaN-preserved matrix.

In [ ]:
POS_WEIGHT = (y == 0).sum() / (y == 1).sum()
print(f"scale_pos_weight: {POS_WEIGHT:.2f}")

LGB_BASE_PARAMS = {
    "objective": "binary",
    "metric": "auc",
    "learning_rate": 0.05,
    "num_leaves": 31,
    "feature_fraction": 0.9,
    "bagging_fraction": 0.9,
    "bagging_freq": 5,
    "min_child_samples": 20,
    "scale_pos_weight": POS_WEIGHT,
    "verbose": -1,
    "n_jobs": -1,
    "seed": SEED,
}

In [ ]:
def fit_lgb_cv(X, y, splits, params, num_boost_round=1000, early_stopping=50, verbose=False):
    """Fit LightGBM per fold with early stopping on val AUC."""
    oof = np.zeros(len(y))
    fold_aucs = []
    fold_aps = []
    fold_iters = []
    t0 = time.time()
    
    for i, (tr_idx, va_idx) in enumerate(splits):
        X_tr, X_va = X[tr_idx], X[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]
        
        train_set = lgb.Dataset(X_tr, label=y_tr)
        val_set = lgb.Dataset(X_va, label=y_va, reference=train_set)
        
        callbacks = [lgb.early_stopping(early_stopping, verbose=False)]
        if verbose:
            callbacks.append(lgb.log_evaluation(period=50))
        
        model = lgb.train(
            params,
            train_set,
            num_boost_round=num_boost_round,
            valid_sets=[val_set],
            callbacks=callbacks,
        )
        proba = model.predict(X_va, num_iteration=model.best_iteration)
        oof[va_idx] = proba
        auc = roc_auc_score(y_va, proba)
        ap = average_precision_score(y_va, proba)
        fold_aucs.append(auc)
        fold_aps.append(ap)
        fold_iters.append(model.best_iteration)
        if verbose:
            print(f"  Fold {i+1}: AUC={auc:.4f}  AP={ap:.4f}  "
                  f"best_iter={model.best_iteration}  elapsed={time.time()-t0:.0f}s")
    
    return oof, fold_aucs, fold_aps, fold_iters

print("Fitting LightGBM baseline...")
lgb_base_oof, lgb_base_aucs, lgb_base_aps, lgb_base_iters = fit_lgb_cv(
    X_lgb, y, splits, LGB_BASE_PARAMS, num_boost_round=2000, verbose=True
)

print(f"\nLightGBM (untuned) CV ROC-AUC: {np.mean(lgb_base_aucs):.4f} (+/- {np.std(lgb_base_aucs):.4f})")
print(f"LightGBM (untuned) CV PR-AUC:  {np.mean(lgb_base_aps):.4f} (+/- {np.std(lgb_base_aps):.4f})")
print(f"LightGBM (untuned) OOF ROC-AUC: {roc_auc_score(y, lgb_base_oof):.4f}")
print(f"LightGBM (untuned) OOF PR-AUC:  {average_precision_score(y, lgb_base_oof):.4f}")
print(f"Average best_iteration across folds: {np.mean(lgb_base_iters):.0f}")

**Expected:** LightGBM (untuned) CV ROC-AUC in the 0.755-0.770 range. The gap over LR is what justifies the interpretability trade-off.

Note `best_iteration` — this tells us roughly how many boosting rounds the optimum is at. If it's saturating near 2000 (our max), we should raise the cap. If it's far below 2000, we have headroom.

## 5. LightGBM hyperparameter tuning with Optuna

**Budget:** 40 trials OR 30 minutes wallclock, whichever comes first. Documented explicitly so a reviewer sees this was a deliberate trade-off, not laziness.

**Search space:**
- `learning_rate`: 0.01 to 0.1 (log scale) — lower with more iterations is the usual lift
- `num_leaves`: 16 to 128 — model complexity
- `feature_fraction`: 0.6 to 1.0 — column sampling
- `bagging_fraction`: 0.6 to 1.0 — row sampling
- `min_child_samples`: 10 to 100 — regularisation against overfitting to rare patterns
- `reg_alpha`, `reg_lambda`: 0.0 to 5.0 — L1 and L2 regularisation

**Objective:** mean ROC-AUC across the 5 folds. Maximise.

**Optimisation:** TPE sampler (Optuna's default — robust on this kind of search space).

In [ ]:
# Use 3-fold inside Optuna for speed - 5-fold * 40 trials = 200 fits is too slow
# After tuning we re-validate the best params on the full 5 folds
OPTUNA_FOLDS = 3
optuna_splits = list(StratifiedKFold(n_splits=OPTUNA_FOLDS, shuffle=True, random_state=SEED).split(X_lgb, y))

def objective(trial):
    params = {
        "objective": "binary",
        "metric": "auc",
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 16, 128),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.6, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.6, 1.0),
        "bagging_freq": 5,
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 100),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 5.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 5.0),
        "scale_pos_weight": POS_WEIGHT,
        "verbose": -1,
        "n_jobs": -1,
        "seed": SEED,
    }
    
    fold_aucs = []
    for tr_idx, va_idx in optuna_splits:
        train_set = lgb.Dataset(X_lgb[tr_idx], label=y[tr_idx])
        val_set = lgb.Dataset(X_lgb[va_idx], label=y[va_idx], reference=train_set)
        
        model = lgb.train(
            params,
            train_set,
            num_boost_round=2000,
            valid_sets=[val_set],
            callbacks=[lgb.early_stopping(50, verbose=False)],
        )
        proba = model.predict(X_lgb[va_idx], num_iteration=model.best_iteration)
        fold_aucs.append(roc_auc_score(y[va_idx], proba))
    
    return np.mean(fold_aucs)

In [ ]:
# Run the search - this is the longest cell in the notebook
optuna.logging.set_verbosity(optuna.logging.WARNING)

study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEED),
)

print("Starting Optuna search (40 trials / 30 min cap)...")
t0 = time.time()
study.optimize(objective, n_trials=40, timeout=1800, show_progress_bar=True)

print(f"\nCompleted in {time.time()-t0:.0f}s")
print(f"Trials run: {len(study.trials)}")
print(f"Best CV ROC-AUC (3-fold): {study.best_value:.4f}")
print(f"Best params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

In [ ]:
# Inspect the top trials
trials_df = study.trials_dataframe().sort_values("value", ascending=False).head(10)
trials_df[["number", "value", "datetime_start", "duration"]].head(10)

## 6. Re-validate the best params on the full 5-fold CV

Tuning used 3-fold for speed. Now we re-fit with the best params on the 5-fold splits to get the headline number we report.

In [ ]:
best_params = {
    "objective": "binary",
    "metric": "auc",
    **study.best_params,
    "bagging_freq": 5,
    "scale_pos_weight": POS_WEIGHT,
    "verbose": -1,
    "n_jobs": -1,
    "seed": SEED,
}

print("Re-fitting LightGBM with tuned params on full 5-fold CV...")
lgb_tuned_oof, lgb_tuned_aucs, lgb_tuned_aps, lgb_tuned_iters = fit_lgb_cv(
    X_lgb, y, splits, best_params, num_boost_round=3000, verbose=True
)

print(f"\nLightGBM (tuned) CV ROC-AUC: {np.mean(lgb_tuned_aucs):.4f} (+/- {np.std(lgb_tuned_aucs):.4f})")
print(f"LightGBM (tuned) CV PR-AUC:  {np.mean(lgb_tuned_aps):.4f} (+/- {np.std(lgb_tuned_aps):.4f})")
print(f"LightGBM (tuned) OOF ROC-AUC: {roc_auc_score(y, lgb_tuned_oof):.4f}")
print(f"LightGBM (tuned) OOF PR-AUC:  {average_precision_score(y, lgb_tuned_oof):.4f}")
print(f"Average best_iteration: {np.mean(lgb_tuned_iters):.0f}")

## 7. Headline comparison

In [ ]:
summary = pd.DataFrame([
    {
        "model": "Logistic Regression",
        "cv_roc_auc_mean": np.mean(lr_aucs),
        "cv_roc_auc_std": np.std(lr_aucs),
        "cv_pr_auc_mean": np.mean(lr_aps),
        "cv_pr_auc_std": np.std(lr_aps),
        "oof_roc_auc": roc_auc_score(y, lr_oof),
        "oof_pr_auc": average_precision_score(y, lr_oof),
    },
    {
        "model": "LightGBM (untuned)",
        "cv_roc_auc_mean": np.mean(lgb_base_aucs),
        "cv_roc_auc_std": np.std(lgb_base_aucs),
        "cv_pr_auc_mean": np.mean(lgb_base_aps),
        "cv_pr_auc_std": np.std(lgb_base_aps),
        "oof_roc_auc": roc_auc_score(y, lgb_base_oof),
        "oof_pr_auc": average_precision_score(y, lgb_base_oof),
    },
    {
        "model": "LightGBM (tuned)",
        "cv_roc_auc_mean": np.mean(lgb_tuned_aucs),
        "cv_roc_auc_std": np.std(lgb_tuned_aucs),
        "cv_pr_auc_mean": np.mean(lgb_tuned_aps),
        "cv_pr_auc_std": np.std(lgb_tuned_aps),
        "oof_roc_auc": roc_auc_score(y, lgb_tuned_oof),
        "oof_pr_auc": average_precision_score(y, lgb_tuned_oof),
    },
])
summary_display = summary.copy()
for c in summary_display.columns:
    if c != "model":
        summary_display[c] = summary_display[c].round(4)
summary_display

**Read the table carefully.** Three things matter:

1. **LightGBM (tuned) vs LR:** how much lift is gradient boosting buying you? In published Home Credit application-only solutions this gap is typically 0.02-0.04 ROC-AUC. That's the cost of LR's interpretability.
2. **LightGBM (tuned) vs (untuned):** did tuning pay off? If the lift is <0.005, that's a flat search and we report it honestly — tuning didn't change much, the defaults were already strong.
3. **CV std:** a model with high std across folds is unstable. We want std < 0.005 for any model we report. If std blows up, the model is overfitting to fold-specific patterns.

## 8. Save OOF predictions and the final model

The OOF predictions are what `04_evaluation.ipynb` uses. We save them as a parquet alongside the IDs so evaluation can join cleanly.

The final LightGBM model is retrained on the full train set with the best params and saved to `models/`. This is what `05_interpretation.ipynb` runs SHAP against.

In [ ]:
# OOF predictions
oof_df = pd.DataFrame({
    "SK_ID_CURR": ids,
    "TARGET": y,
    "lr_pred": lr_oof,
    "lgb_base_pred": lgb_base_oof,
    "lgb_tuned_pred": lgb_tuned_oof,
})
oof_path = PROCESSED_DIR / "oof_predictions.parquet"
oof_df.to_parquet(oof_path, index=False)
print(f"Saved OOF predictions: {oof_path}")
print(oof_df.head(5))

In [ ]:
# CV summary table
summary_path = PROCESSED_DIR / "cv_summary.parquet"
summary.to_parquet(summary_path, index=False)
print(f"Saved CV summary: {summary_path}")

In [ ]:
# Retrain final LightGBM on all of train (no holdout) using average best_iteration
final_iter = int(np.mean(lgb_tuned_iters))
print(f"Retraining final LightGBM with {final_iter} iterations on full train set...")

final_train_set = lgb.Dataset(X_lgb, label=y)
final_model = lgb.train(
    best_params,
    final_train_set,
    num_boost_round=final_iter,
    callbacks=[],
)

model_path = MODELS_DIR / "lightgbm_final.txt"
final_model.save_model(str(model_path))
print(f"Saved final LightGBM model: {model_path}")

# Also save the params and feature column order for interpretation notebook
import json
metadata = {
    "best_params": study.best_params,
    "scale_pos_weight": float(POS_WEIGHT),
    "final_num_boost_round": final_iter,
    "feature_cols": FEATURE_COLS,
    "cv_oof_roc_auc": float(roc_auc_score(y, lgb_tuned_oof)),
    "cv_oof_pr_auc": float(average_precision_score(y, lgb_tuned_oof)),
}
metadata_path = MODELS_DIR / "lightgbm_metadata.json"
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)
print(f"Saved metadata: {metadata_path}")

## 9. Summary

### What was built
- LR baseline fitted with per-fold scaling and balanced class weights
- LightGBM baseline with sensible defaults (`scale_pos_weight` for imbalance)
- LightGBM tuned via Optuna (40 trials / 30 min cap) on 3-fold inner CV
- Best tuned model re-validated on the full 5-fold CV
- OOF predictions saved for all three models
- Final LightGBM model retrained on full train set, saved to `models/`

### Defensible choices
1. **Stratified k-fold, fixed seed** — folds are reproducible and class-balanced
2. **All preprocessing inside the fold loop** — no leakage from validation into train
3. **3-fold inside Optuna, 5-fold for headline numbers** — pragmatic speed/stability trade-off
4. **Class weighting on both models** — explicit handling of 8% positive class
5. **Bounded tuning budget** — 40 trials / 30 min, documented in markdown
6. **Both untuned and tuned LightGBM reported** — quantifies whether tuning actually paid off

### What `04_evaluation.ipynb` does next
- Load the OOF predictions
- ROC and PR curves comparing all three models
- KS statistic
- Calibration plot
- Threshold sweep with operating-point interpretation
- Business framing carefully scoped to historical observed distribution (no reject inference claims)
